In [ ]:
import rasterio as rs
import numpy as np
from rasterio.enums import Resampling
from rasterio.windows import Window
import pandas as pd
import numpy as np

In [ ]:
from affine import Affine
import rasterio.transform

In [ ]:
!pip install pyproj

In [ ]:
from pyproj import Transformer

In [ ]:
!pip install rasterio

In [ ]:
patcheslist=[]
pixeltoworldcoordinates=[]

In [ ]:
with rs.open("T44RLT_20230225T051801_B04_10m.jp2") as f1,rs.open("T44RLT_20230225T051801_B08_10m.jp2") as f2,rs.open("T44RLT_20230225T051801_B12_20m.jp2") as f12,rs.open("T44RLT_20230225T051801_B11_20m.jp2") as f11:
  transform=f1.transform
  transformer = rasterio.transform.AffineTransformer(transform)
  for row in range (0,11200,224):
    for col in range (0,11200,224):
      B04 = f1.read(1, window=Window(col, row, 224, 224),boundless=True,fill_value=0)
      B08 = f2.read(1, window=Window(col, row, 224, 224),boundless=True,fill_value=0)
      B08=B08.astype(np.float32)
      B04=B04.astype(np.float32)
      B11=f11.read(1,window=Window(col//2, row//2, 112, 112),boundless=True,fill_value=0,out_shape=(1, 224, 224),resampling=Resampling.bilinear)
      B12=f12.read(1,window=Window(col//2, row//2, 112, 112),boundless=True,fill_value=0,out_shape=(1, 224, 224),resampling=Resampling.bilinear)
      B11=np.squeeze(B11)
      B12=np.squeeze(B12)
      B11=B11.astype(np.float32)
      B12=B12.astype(np.float32)
      with np.errstate(divide='ignore', invalid='ignore'):
        NDVI= (B08 - B04) / (B08 + B04)
        NBR=(B08-B12)/(B08+B12)
      arraystoinput=[B04,B08,B11,B12,NDVI,NBR]
      cnninput=np.stack(arraystoinput,axis=0)
      patcheslist.append(cnninput)
      x_min,y_max=transformer.xy(row, col, offset='ul')
      x_max, y_min=transformer.xy(row+224, col+224, offset='lr')
      onetuple=(x_min,y_min,x_max,y_max)
      pixeltoworldcoordinates.append(onetuple)


In [ ]:
len(patcheslist)

2500

In [ ]:
print(len(patcheslist))
print(len(pixeltoworldcoordinates))
print(patcheslist[0].shape)

2500
2500
(6, 224, 224)


In [ ]:
x_tl,y_tl=transformer.xy(0, 0, offset='ul')
x_br,y_br=transformer.xy(10980, 10980, offset='lr')

In [ ]:
transformer_pyproj = Transformer.from_crs("EPSG:32644", "EPSG:4326",always_xy=True)
long_tl, lat_tl = transformer_pyproj.transform(x_tl,y_tl)
long_br,lat_br=transformer_pyproj.transform(x_br,y_br)


In [ ]:
print(lat_tl, long_tl)
print(lat_br, long_br)

29.814255053135902 78.93040991513051
28.836207549516807 80.07549933025473
